# Incremental Embedding Generation via Endpoint API

**Resource-Optimized Workflow**

This notebook generates embeddings for pending plays using the Droplet's FastAPI endpoints.

**Architecture:**
- **Colab**: Handles heavy embedding generation (GPU accelerated)
- **Droplet**: Provides lightweight endpoints for data fetch and integration

**Endpoints:**
1. `GET /api/embeddings/pending` - Fetch plays without embeddings
2. `GET /api/embeddings/pca-model` - Download PCA transformer
3. `POST /api/embeddings/integrate` - Upload and integrate new embeddings

**Requirements:**
- Enable GPU runtime (Runtime → Change runtime type → T4 GPU)
- Run all cells in sequence

In [ ]:
# Install dependencies
!pip install -q sentence-transformers requests joblib

In [ ]:
import torch

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("\n⚠️  WARNING: GPU not available. Embedding generation will be slow.")
    print("   Enable GPU: Runtime → Change runtime type → Hardware accelerator → GPU")

In [ ]:
# Configuration
API_URL = "http://cratemusic.duckdns.org"  # Temporarily using HTTP (SSL will be restored)
MODEL_NAME = 'sentence-transformers/multi-qa-mpnet-base-dot-v1'
BATCH_SIZE = 1024 if device == 'cuda' else 32
FETCH_LIMIT = 1000  # Process in batches of 1000

print(f"API URL: {API_URL}")
print(f"Model: {MODEL_NAME}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Fetch limit: {FETCH_LIMIT}")

## Step 1: Fetch Pending Plays

In [ ]:
import requests
import json

print(f"Fetching pending plays from {API_URL}/api/embeddings/pending...\n")

response = requests.get(f"{API_URL}/api/embeddings/pending?limit={FETCH_LIMIT}")
response.raise_for_status()

batch = response.json()

print(f"{'='*80}")
print(f"BATCH RECEIVED")
print(f"{'='*80}")
print(f"Batch ID: {batch['batch_id']}")
print(f"Plays to embed: {len(batch['plays'])}")
print(f"Total pending: {batch['total_pending']}")
print(f"\nSample play:")
print(json.dumps(batch['plays'][0], indent=2))

## Step 2: Download PCA Transformer

In [ ]:
import joblib

print("Downloading PCA transformer...")

pca_response = requests.get(f"{API_URL}/api/embeddings/pca-model")
pca_response.raise_for_status()

with open('pca_transformer_256d.joblib', 'wb') as f:
    f.write(pca_response.content)

print(f"✓ Downloaded PCA transformer ({len(pca_response.content) / 1e3:.1f} KB)")

# Load PCA transformer
pca = joblib.load('pca_transformer_256d.joblib')
print(f"✓ Loaded PCA transformer: {pca.n_components} components")

## Step 3: Generate Embeddings (GPU Accelerated)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm
import time

# Load model
print(f"Loading model: {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME, device=device)
print(f"✓ Model loaded (embedding dim: {model.get_sentence_embedding_dimension()})\n")

# Extract texts and IDs
texts = [play['enriched_text'] for play in batch['plays']]
play_ids = [play['id'] for play in batch['plays']]

print(f"{'='*80}")
print(f"GENERATING EMBEDDINGS")
print(f"{'='*80}")
print(f"Texts: {len(texts)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Device: {device}\n")

start_time = time.time()

# Step 1: Generate 768d embeddings (normalized)
print("Step 1: Generating 768d embeddings...")
embeddings_768d = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,  # ← CRITICAL: L2 normalize for cosine similarity
    convert_to_numpy=True,
    device=device
)

elapsed_768d = time.time() - start_time
print(f"✓ Generated 768d embeddings: {embeddings_768d.shape}")
print(f"  Time: {elapsed_768d:.1f}s ({len(texts) / elapsed_768d:.1f} texts/sec)")
print(f"  Memory: ~{embeddings_768d.nbytes / 1e6:.1f} MB\n")

## Step 4: Apply PCA Dimensionality Reduction

In [ ]:
# Step 2: Apply PCA (768d → 256d)
print("Step 2: Applying PCA transformation (768d → 256d)...")
embeddings_256d = pca.transform(embeddings_768d)
print(f"✓ Reduced to 256d: {embeddings_256d.shape}")

# Step 3: Re-normalize (PCA changes norms)
print("Step 3: Re-normalizing 256d embeddings...")
for i in range(len(embeddings_256d)):
    norm = np.linalg.norm(embeddings_256d[i])
    if norm > 0:
        embeddings_256d[i] = embeddings_256d[i] / norm

embeddings_256d = embeddings_256d.astype('float32')
print(f"✓ Normalized 256d embeddings")
print(f"  Memory: ~{embeddings_256d.nbytes / 1e6:.1f} MB")

# Free GPU memory
if device == 'cuda':
    del embeddings_768d
    torch.cuda.empty_cache()
    print("✓ Freed GPU memory")

total_time = time.time() - start_time
print(f"\n{'='*80}")
print(f"EMBEDDING GENERATION COMPLETE")
print(f"{'='*80}")
print(f"Total time: {total_time:.1f}s")
print(f"Speed: {len(texts) / total_time:.1f} texts/sec")
print(f"Final shape: {embeddings_256d.shape}")

## Step 5: Compute Checksum

In [ ]:
import hashlib

embeddings_bytes = embeddings_256d.tobytes()
checksum = hashlib.sha256(embeddings_bytes).hexdigest()

print(f"Checksum: sha256:{checksum}")
print(f"Payload size: {len(embeddings_bytes) / 1e6:.1f} MB")

## Step 6: Encode and POST to Integration Endpoint

In [ ]:
import base64
from datetime import datetime

# Encode to base64
print("Encoding embeddings to base64...")
embeddings_b64 = base64.b64encode(embeddings_bytes).decode('utf-8')
print(f"✓ Encoded to base64 ({len(embeddings_b64) / 1e6:.1f} MB)")

# Prepare payload
payload = {
    "batch_id": batch['batch_id'],
    "play_ids": play_ids,
    "embeddings_256d_b64": embeddings_b64,
    "checksum": f"sha256:{checksum}",
    "metadata": {
        "generated_by": "colab",
        "model_name": MODEL_NAME,
        "generation_time": datetime.utcnow().isoformat(),
        "device": device,
        "total_plays": len(play_ids)
    }
}

payload_size = len(json.dumps(payload)) / 1e6
print(f"Payload size: {payload_size:.1f} MB\n")

# POST to integration endpoint
print(f"{'='*80}")
print(f"POSTING TO INTEGRATION ENDPOINT")
print(f"{'='*80}")
print(f"URL: {API_URL}/api/embeddings/integrate")
print(f"Payload size: {payload_size:.1f} MB")
print(f"Timeout: 300s (5 minutes)\n")

print("Uploading...")
start_upload = time.time()

response = requests.post(
    f"{API_URL}/api/embeddings/integrate",
    json=payload,
    headers={"Content-Type": "application/json"},
    timeout=300
)

upload_time = time.time() - start_upload
print(f"✓ Upload complete ({upload_time:.1f}s)\n")

response.raise_for_status()
result = response.json()

print(f"{'='*80}")
print(f"INTEGRATION COMPLETE")
print(f"{'='*80}")
print(json.dumps(result, indent=2))

## Summary Statistics

In [ ]:
if result['status'] == 'success':
    print(f"\n{'='*80}")
    print(f"SUCCESS!")
    print(f"{'='*80}")
    print(f"✓ Integrated: {result['integration']['plays_integrated']} plays")
    print(f"✓ Total embeddings: {result['integration']['total_embeddings_after']:,}")
    print(f"✓ Pending remaining: {result['integration']['pending_plays_remaining']:,}")
    print(f"✓ Index rebuilt: {result['index']['rebuilt']}")
    
    if result['integration']['pending_plays_remaining'] > 0:
        print(f"\n⚠️  {result['integration']['pending_plays_remaining']} plays still pending.")
        print(f"   Run this notebook again to process the next batch.")
    else:
        print(f"\n🎉 All plays have embeddings!")
else:
    print(f"\n{'='*80}")
    print(f"ERROR")
    print(f"{'='*80}")
    print(f"✗ Error code: {result['error_code']}")
    print(f"  Message: {result['message']}")

## Notes

**Pipeline Overview:**
1. Fetch pending plays from Droplet (lightweight JSON)
2. Download PCA transformer (775KB, cached)
3. Generate 768d embeddings in Colab (GPU accelerated)
4. Apply PCA dimensionality reduction (768d → 256d)
5. Re-normalize embeddings (preserves cosine similarity)
6. Upload to Droplet for integration (streaming base64)

**Memory Usage:**
- **Colab**: ~500MB peak (fits in free tier)
- **Droplet**: <100MB overhead (stays under 3GB total)

**Performance:**
- T4 GPU: ~500-800 texts/sec
- CPU: ~20-50 texts/sec
- 1000 plays: ~2-3 minutes on GPU, ~20-50 minutes on CPU

**For Large Batches:**
If processing all 1,431 pending plays:
1. Increase `FETCH_LIMIT` to 1431 (or keep at 1000 and run twice)
2. Ensure GPU runtime is enabled
3. Monitor Droplet memory during integration

**Troubleshooting:**
- If timeout: Reduce `FETCH_LIMIT` to process smaller batches
- If memory error: Check Droplet stats with `docker stats kexp-search-api`
- If checksum mismatch: Verify PCA transformer version matches
